# Analyse de la géométrie d'étalement d'une cellule

_Version pour une cellule s'étalant sur une seule plaque_

## 1. Imports et paramètres graphiques

In [ ]:
%matplotlib inline

import os
import time
import math
import numpy as np
import skimage
import tifffile
import matplotlib.pyplot as plt
from skimage import img_as_ubyte
from scipy import optimize
from skimage import data, filters, io, color, exposure
from skimage.transform import hough_line, hough_line_peaks
from skimage.segmentation import active_contour
from skimage import img_as_float

plt.rcParams['font.size'] = 14
plt.rcParams['figure.max_open_warning'] = 50
plt.rcParams['image.cmap'] = 'gray'

# print(plt.rcParams)

## 2. Fonctions

Définition de l'ensemble des fonctions utiles, qui seront ensuite appelée dans les scripts de la partie 3.

### 2.1. Lecture & écriture dans les fichiers

In [ ]:
def create_dir(path):
    """Vérifie si un dossier existe déjà à l'emplacement path et sinon, le crée."""
    if not os.path.exists(path):
        os.makedirs(path)

def writeSnakeInFile(S, dirname, filename, header = True):
    """
    Ecrit un snake au format TYX (numpy array) dans un fichier texte, avec ou sans ligne d'en-tête.
    
    Remarque : les snakes initiaux n'ont pas d'en-tête (ils sont fait avec imageJ) ; 
    Les snakes résultats ont un en-tête (pour être lisible avec un tableur).
    """
    # Convention dans ce script : on ne met pas de header dans les fichiers snakes initiaux,
    # et on en met un dans les fichiers snakes résultats. Adapter la valeur du booleen 'header' en conséquence
    # (en écriture ET en lecture)
    nt, nbpts = S.shape[0], S.shape[1]
    try:
        snakeFile = open(os.path.join(dirname, filename), "w")
        if header:
            for i in range(nt):
                snakeFile.write("Y(" + str(i) + ",:)" + "\t")
                snakeFile.write("X(" + str(i) + ",:)" + "\t")
                if i == nt-1:
                    snakeFile.write("\n")
        for j in range(nbpts):
            for i in range(nt):
                snakeFile.write(str(S[i,j,0]) + "\t")
                snakeFile.write(str(S[i,j,1]) + "\t")
                if i == nt-1:
                    snakeFile.write("\n")
    finally:
        snakeFile.close()


def extractSnakeFromFile(dirname, snakeFileName, header = True, scale = 1, reverseXY = False):
    """
    Lit un snake écrit dans un fichier .txt avec la fonction writeSnakeInFile.
    'header' permet de lire un fichier avec ou sans en-tête ; 
    'scale' multiplie toutes les valeurs importées par une constante (pour convertir entre µm et px) ;
    'reverseXY' permet au choix d'inverser ou non l'ordre dans lequel on lit les coordonnées.
    
    Remarques
    (1) Les snakes venant d'imageJ sont au format XY, en µm, et n'ont pas d'en-tête.
    Dans ce code, on va tout mettre en format YX et en pixel pour utiliser les fonctions du module skimage.
    On utilise donc header,  scale et reverseXY lorsqu'on importe les coordonnées d'un snake tracé en imageJ, pour repasser ce dernier au format avec lequel on travaille (YX, pixels).
    (2) Les snakes résultats ont un header, sont en pixels, au format YX. A priori pas besoin des arguments optionnels dans ce cas.
    """

    snakePath = dirname + snakeFileName
    try:
        snakeFile = open(snakePath, 'r')
        snakeLines = snakeFile.readlines()
        snakeNbpts = len(snakeLines)-1
        line_0 = snakeLines[0].split('\t')
        nbpic = len(line_0)//2
        snakeRes = np.zeros((nbpic, snakeNbpts, 2), dtype=float)
        firstLineOfData = 0
        if header:
            firstLineOfData += 1
        for j in range(firstLineOfData, firstLineOfData+snakeNbpts):
            line = snakeLines[j].split('\t')
            for i in range(nbpic):
                if not reverseXY:
                    snakeRes[i, j-firstLineOfData, 0] = float(line[2*i]) * scale
                    snakeRes[i, j-firstLineOfData, 1] = float(line[2*i+1]) * scale
                else:
                    snakeRes[i, j-firstLineOfData, 1] = float(line[2*i]) * scale
                    snakeRes[i, j-firstLineOfData, 0] = float(line[2*i+1]) * scale
            # Remarques : les snakes venant d'imageJ sont au format XY et en µm. On va tout mettre en format YX et en pixel pour utiliser les fonctions du module skimage
            # On utilise donc scale et reverseXY lorsqu'on importe les coordonnées d'un snake tracé en imageJ.
            # Pas besoin sinon.      
            
    finally:
        snakeFile.close()
    return(snakeRes)

### 2.2. Prétraitement des images

* Fonctions obsolètes dans la version actuelle du code, mais conservées ici au cas où.

In [ ]:
def makeFilter(image, blur = 4):
    imageBlur = filters.gaussian(image, blur)
    return(imageBlur)


def enhanceContrast(image, Low = 2, High = 98):
    # Contrast stretching
    pLow, pHigh = np.percentile(image, (Low, High))
    imageRescale = exposure.rescale_intensity(image, in_range=(pLow, pHigh))
    return(imageRescale)


def makeEdge(image):
    imageEdge = filters.sobel(image)
    return(imageEdge)
 
    
def getThreshold(image, method = 'isodata'):
    if method == 'isodata':
        threshold = filters.threshold_isodata(image)
    if method == 'otsu':
        threshold = filters.threshold_otsu(image)
    if method == 'li':
        threshold = filters.threshold_li(image)
    if method == 'mean':
        threshold = filters.threshold_mean(image)
    if method == 'minimum':
        threshold = filters.threshold_minimum(image)
    if method == 'local':
        threshold = filters.threshold_local(image, 27)
    return(threshold)
   
    
def makeBinary(image, threshold, invert = False):
    mask = image < threshold
    if not invert:
        imageBinary = 255*np.ones_like(img_as_ubyte(image))
        imageBinary[mask] = 0
    else:
        imageBinary = mask
    return(imageBinary)

* Fonctions auxiliaires du prétraitement actuellement utilisé.

In [ ]:
def houghTransform(rawImage, p = 8, plot = False):
    """
    Détecte automatiquement la ligne de pixels sombre la plus forte de l'image en utilisant la transformée de Hough.
    Renvoie ses coordonnées cartésiennes a et b (où y = ax + b) 
    et cylindriques lineDist et lineAngle (lineDist = x*cos(lineAngle) + y*sin(lineAngle)).
    Parce que les lignes trop épaisses de pixels sombres sont imparfaitement repérées, on érode un peu l'image biarisée avant d'appliquer la transformation de Hough.
    """
    
    # Binarization - we keep only the lowest p percentile of the image
    threshold = np.percentile(rawImage, p)
    binaryImage = makeBinary(rawImage, threshold, invert = True) # Invert = true to have the pixel below the threshold in white
    
    # Use hough transform ; source: https://scikit-image.org/docs/dev/api/skimage.transform.html?highlight=hough#skimage.transform.hough_line
    hspace, angles, distances = hough_line(binaryImage)
    accum, lineAngle, lineDist = hough_line_peaks(hspace, angles, distances, num_peaks=1) # num_peaks = 1 ensure we take only the best line, which usually corresponds to the plate
    
    # Amélioration en érodant les lignes
    count = 0
    better_lineAngle, better_lineDist = lineAngle, lineDist
    binaryImageErode = binaryImage[:][:]
    # figTest, axTest = plt.subplots(3,1, figsize = (10, 25))
    
    while count < 4 and abs((lineAngle-better_lineAngle)/lineAngle) < 0.1 and abs((lineDist-better_lineDist)/lineDist) < 0.05:
        # Tant que l'érosion ne donne pas un résultat très différent, on suppose qu'il va être un peu meilleur. 
        # On répète maximum 3 fois.
        lineAngle, lineDist = better_lineAngle, better_lineDist
        binaryImageErode = skimage.morphology.binary_erosion(binaryImageErode)
        # axTest[count].imshow(binaryImageErode)
        hs, ang, d = hough_line(binaryImageErode)
        acc, better_lineAngle, better_lineDist = hough_line_peaks(hs, ang, d, num_peaks=1)
        count += 1
    
    # Transform the polar coordinates back to the cartesian ones
    b = float(lineDist / np.sin(lineAngle))
    a = float(- 1 / np.tan(lineAngle))
    
    # if plot = true we can plot the resulting line
    if plot:
        ny = rawImage.shape[0]
        nx = rawImage.shape[1]
        fig, ax = plt.subplots(1,3, figsize = (20, 5))
        ax[0].imshow(rawImage)
        ax[1].imshow(binaryImage)
        ax[2].imshow(hspace)
        ax[0].plot([0, nx-1], [b, a*(nx-1)+b], 'r-')
        ax[1].plot([0, nx-1], [b, a*(nx-1)+b], 'r-')
        # ax[2].plot([lineAngle*180/np.pi], [lineDist], 'ro') # Ne marche pas
    return(a, b, lineAngle, lineDist)


def makeHorizontal(imageStack, theta, SnakeInit_YX = np.array([])):
    """
    Applique une rotation à un stack d'image d'angle theta, centrée sur le centre des images, puis les croppe d'une marge de pixel 'margin' pour enlever les coins noirs créés par la rotation.
    En prenant comme 'theta' l'angle trouvé par la fonction 'houghTransform', ceci permet d'avoir la lame de verre horizontale dans toutes les images de la stack
    Si on entre un array non vide comme SnakeInit_YX, la fonction fait subir à ce snake la même rotation qu'aux images.
    Comme ça on peut tracer le snake initial sous imageJ sur l'image de base, non tournée.
    """
    
    # Etape 1 : rotation dans un stack d'image pour rendre horizontale la lame de verre
    # détectée par une transformée de Hough.
    n_images = imageStack.shape[0]
    nx = imageStack.shape[2]
    ny = imageStack.shape[1]
    imageListResult = []
    T = skimage.transform.EuclideanTransform(
        translation = (-nx/2, -ny/2)
        )
    thetaBis = np.sign(theta)*math.pi/2 - theta # theta c'est l'angle qui définit la droite de la lamelle en coordonnées polaires.
                                                # thetaBis c'est l'angle de rotation à appliquer à l'image pour la redresser (décalage de + ou - pi/2 en gros).
    R = skimage.transform.EuclideanTransform(
        rotation = thetaBis 
        )
    Xmargin = int(abs(nx - (nx*np.cos(thetaBis) - ny*np.sin(thetaBis)))/2) + 2 # La marge en X pour couper les coins noirs d'après rotation. + 2 pixels pour être sûr.
    Ymargin = int(abs(ny - (ny*np.cos(thetaBis) + nx*np.sin(thetaBis)))/2) + 2 # La marge en Y pour couper les coins noirs d'après rotation. + 2 pixels pour être sûr.
    matrix = np.linalg.inv(T.params) @ R.params @ T.params
    tform = skimage.transform.EuclideanTransform(matrix)
    for i in range(n_images):
        image_transform = skimage.transform.warp(img_as_float(imageStack[i]), tform.inverse)
        imageListResult.append(image_transform[Ymargin:-Ymargin,Xmargin:-Xmargin])
    
    # Etape 2 : si l'argument SnakeInit_YX n'est pas de longueur 0, la fonction va appliquer la même rotations
    # qu'aux images à la liste de points SnakeInit_YX. Ceci permet de tracer son snake initial sur l'image 'penchée'
    # et de lui appliquer ensuite la même rotation qu'aux images pour qu'il reste utilisable.
    lengthSnake = SnakeInit_YX.shape[0]
    rotatedSnakeInit_YX = np.copy(SnakeInit_YX)
    if lengthSnake != 0:
        tempVect = np.zeros(3)
        tempRotatedVect = np.zeros(3)
        for i in range(lengthSnake):
            # les snakes sont au format R, C donc SnakeInit_YX[0] contient les Y et SnakeInit_YX[1] les X
            tempVect[0] = SnakeInit_YX[i,1]
            tempVect[1] = SnakeInit_YX[i,0]
            tempVect[2] = 1 # On a besoin d'un 3e élément valant 1 pour appliquer l'homographie
            tempRotatedVect = np.dot(tform.params, tempVect)
            rotatedSnakeInit_YX[i, 1] = tempRotatedVect[0] - Xmargin
            rotatedSnakeInit_YX[i, 0] = tempRotatedVect[1] - Ymargin
        return(np.array(imageListResult), rotatedSnakeInit_YX)
    
    else:
        return(np.array(imageListResult))

def denoisePlateShadow(inputImage, referenceSide = 'left&right', addGaussianPlate = True, platePositionFromHough = 0):
    """
    Ne s'applique qu'aux images traitées par la fonction 'makeHorizontal'.
    
    """
    image = skimage.util.img_as_uint(inputImage) # conversion en image 16bits
    nx = image.shape[1]
    ny = image.shape[0]
    listCols = [i for i in range(nx)]
    if referenceSide == 'right':
        selectedCols = listCols[-15:-5]
    elif referenceSide == 'left':
        selectedCols = listCols[5:15]
    elif referenceSide == 'left&right':
        selectedCols = listCols[5:15] + listCols[-15:-5]
    medianOfCols = np.zeros(ny)
    for i in range(ny):
        medianOfCols[i] = np.median(image[i,selectedCols])
    
    globalMax = np.max(medianOfCols)
    plateCenterPos = ny-5
    currentVal = medianOfCols[ny-5]
    currentMin = np.min(medianOfCols[ny-10:ny])
    while not (currentVal == currentMin and currentVal < globalMax/2):
        plateCenterPos = plateCenterPos-1
        currentVal = medianOfCols[plateCenterPos]
        currentMin = np.min(medianOfCols[plateCenterPos-5:plateCenterPos+5])
    shadowProfile = medianOfCols[:ny]
    shadowProfileComplement = np.array([(2**16 - 1) for i in range(ny)]) - shadowProfile # 2**16 = valeur max d'une image 16bit
    
    denoisedImage = np.copy(image)
    
    
    for x in range(nx):
        for y in range(ny):
            denoisedImage[y][x] = min((2**16 - 1), denoisedImage[y][x] + 0.9*shadowProfileComplement[y])
    
    # Gaussian plate. Après avoir "gommé" la lame et son ombre de l'image, on redessine la lame sous la forme d'une ligne horizontale 
    # avec un profil d'intensité gaussien assez piqué dans la direction verticale
    
    if platePositionFromHough == 0:
        m = plateCenterPos
    else:
        m = platePositionFromHough
    s = 1
    A = ((2**16 - 1) - 1*np.min(denoisedImage)) * (s*(2*np.pi)**0.5)
    gauss = lambda y : (0.8*A/(s*(2*np.pi)**0.5))*np.exp(-(y-m)**2/(2*s*s))
    
    for x in range(nx):
        for y in range(ny):        
            denoisedImage[y][x] -= addGaussianPlate * min(denoisedImage[y][x], gauss(y))
            
    p1, p99 = np.percentile(denoisedImage, (1, 99))
    denoisedImage = exposure.rescale_intensity(denoisedImage, in_range=(p1, p99))

    return(denoisedImage)

* Fonction principale du prétraitement.

In [ ]:
def pretraitement(imageStack, SnakeInit_YX = np.array([]), referenceSide = 'left&right', 
                  saveDirPath = '', saveFileName = ''):
    # Chronométrer la fonction !
    Ti = time.time()
    
    n_images = imageStack.shape[0]
    nx = imageStack.shape[2]
    ny = imageStack.shape[1]
    image_0 = imageStack[0]
    
    # Détecte la postion et l'angle de la ligne sombre lame de verre avec une transformée de Hough.
    a, b, lineAngle, lineDist = houghTransform(image_0)
    
    # Redresse les images (et l'éventuel snake initial) pour avoir la lame de verre à l'horizontale (nécessaire pour le denoising).
    SnakeInitIsEmpty = (SnakeInit_YX.shape[0] == 0)
    if not SnakeInitIsEmpty:
        horizontalImageStack, rotatedSnakeInit_YX = makeHorizontal(imageStack, lineAngle, SnakeInit_YX)
    else:
        horizontalImageStack = makeHorizontal(imageStack, lineAngle, SnakeInit_YX)
    
    # Denoising des images en "soustrayant l'ombre de la lame" colonne par colonne.   
    denoisedImageList = []
    for i in range(n_images):
        denoisedImageList.append(denoisePlateShadow(horizontalImageStack[i], referenceSide, platePositionFromHough = b))
    denoisedImageStack = np.array(denoisedImageList)
    
    # Sauvegarde les images traitées si un emplacement de sauvegarde a été entré en argument.
    if len(saveDirPath) != 0 and len(saveFileName) != 0:
        tifffile.imwrite(saveDirPath + saveFileName + '.tif', denoisedImageStack, metadata={'axes': 'TYX'})
        if not SnakeInitIsEmpty:
            writeSnakeInFile(np.array([rotatedSnakeInit_YX]), saveDirPath, saveFileName + '_rotatedSnakeInit' + '.txt', header = False)
    
    #Fin du chrono
    Tf = time.time()
    print(Tf - Ti)
    
    # Renvoie les images traitées, et l'éventuel snake initial retouché.
    if not SnakeInitIsEmpty:
        return(denoisedImageStack, rotatedSnakeInit_YX)
    else:
        return(denoisedImageStack)

### 2.3. Fonction principale : tracé des snakes

In [ ]:
def makeSnake(A = 0.1, B = 15 , W_L = -50, W_E = 400, B_C = 'fixed', Max_It = 100, blur = 2, SnakePoints = 100, stepPlot = 5, showPlot = True):
    """
    Trace un snake itératif autour de la cellule dans une séquence temporelle.
    """
    
    # Chronométrer la fonction !
    Ti = time.time()
    
    #
    resultsSubdirName = 'Results_' + imageFileName
    resultsPath = mainDirPath + resultsDirName + resultsSubdirName + "//quantitative_results//"
    exampleImagePath = mainDirPath + resultsDirName + resultsSubdirName + "//exampleImages//"
    exampleImageName = resultsSubdirName + '_image_'
    exampleImageExt = '.png'
    
    # Créer le dossier de destination des résultats
    create_dir(mainDirPath + resultsDirName)
    create_dir(mainDirPath + resultsDirName + resultsSubdirName)
    create_dir(resultsPath)
    create_dir(exampleImagePath)

    # Extraire SnakeInit du fichier texte
    SnakeInit_path = mainDirPath + sourceDirName + SnakeInit_filename
    SnakeInit_file = open(SnakeInit_path, 'r')
    SnakeInit_lines = SnakeInit_file.readlines()
    nS_raw = len(SnakeInit_lines)
    # print(nS_raw)
    
     # On prend seulement une sous-partie des points - A corriger
    stepSnake = int(nS_raw / SnakePoints)
    n_Snake = nS_raw // stepSnake

    SnakeInit_X = np.zeros(n_Snake)
    SnakeInit_Y = np.zeros(n_Snake)
    
    for i in range(0, n_Snake):
        line = SnakeInit_lines[i*stepSnake].split('\t')
        
        SnakeInit_X[i] = int(float(line[1]))
        SnakeInit_Y[i] = int(float(line[0]))
    SnakeInit_YX = np.array([SnakeInit_Y, SnakeInit_X]).T
    
    # Ouvrir l'image et lire sa taille.
    # Rq: io.imread() ouvre l'image indexée en coordonnées rows, columns ; donc les coordonnées sont dans l'ordre [T, R, C] soit y, x].
    imagePath = mainDirPath + sourceDirName + imageFileName + ext
    imageStack = io.imread(imagePath)
    nt = imageStack.shape[0]
    nx = imageStack.shape[2] # Cf remarque ci-dessus : nx est bien dans shape[2]
    ny = imageStack.shape[1] # et ny dans shape[1]
    
    # Initialiser la liste des snakes résultats
    SnakeResult = np.zeros((nt, n_Snake, 2))
     
    # BOUCLE PRINCIPALE
    for i in range(nt):
        # On lit la ieme image, on lui applique le prétraitement pour la binariser, puis on la floute un peu.
        image_i = imageStack[i]
        imageBlur_i = filters.gaussian(image_i, blur)
        #imageBlur_i = image_i[:][:]
        
        # Appels à la fonction active_contour : c'est ici qu'on construit le snake proprement dit
        SnakeCurrent_YX = active_contour(imageBlur_i, SnakeInit_YX, alpha = A/SnakePoints, beta = B/SnakePoints, gamma=0.01, w_line = W_L/SnakePoints, w_edge = W_E/SnakePoints, boundary_condition=B_C, max_iterations = Max_It, coordinates = 'rc') # coordinates = 'xy'
        
        # Stockage du snake pour le bord
        SnakeResult[i] = SnakeCurrent_YX
                
        # Afficher / enregistrer l'image fittée par les snake. On n'en affiche qu'1 toutes les stepPlot images.
        if i == 1 or i%stepPlot == 0:
            
            fig, ax = plt.subplots(1, 2, sharey=True, figsize=(20, 5))
            
            ax[0].imshow(imageBlur_i)
            ax[0].plot(SnakeInit_YX[:, 1], SnakeInit_YX[:, 0], '--b', lw=2)
            ax[0].plot(SnakeCurrent_YX[:, 1], SnakeCurrent_YX[:, 0], '-r', lw=2)
            # ax[0].set_xticks([]), ax[0].set_yticks([])
            ax[0].axis([0, imageBlur_i.shape[1], imageBlur_i.shape[0], 0])
            ax[0].set_title("Snake de l'image " + str(i) + " du film '" + imageFileName + "'")
            
            ax[1].imshow(image_i)
            ax[1].plot(SnakeInit_YX[:, 1], SnakeInit_YX[:, 0], '--b', lw=2)
            ax[1].plot(SnakeCurrent_YX[:, 1], SnakeCurrent_YX[:, 0], '-r', lw=2)
            # ax[1].set_xticks([]), ax[1].set_yticks([])
            ax[1].axis([0, imageBlur_i.shape[1], imageBlur_i.shape[0], 0])
            ax[1].set_title("...et sur l'image originale")
            
            currentExampleImagePath = exampleImagePath + exampleImageName + str(i) + exampleImageExt
            fig.savefig(currentExampleImagePath)
            
            if not showPlot:
                plt.close('all')
    
        # Mettre à jours les conditions initiales pour l'image suivante !
        SnakeInit_YX = SnakeCurrent_YX

    #Export des résultats dans le dossier 'results'
    #resultsdirname = os.path.join(savedirname, saveresultname, 'results')
    writeSnakeInFile(SnakeResult, resultsPath , "Snake.txt")
    
    #Fin du chrono
    Tf = time.time()
    print(Tf - Ti)
    
    #Export des infos contextuelles dans un .txt
    exportFile = open(mainDirPath + resultsDirName + resultsSubdirName + '//ActiveContourData.txt', "w")
    exportFile.write("Paramètres utilisés\n")
    exportFile.write("\n")
    exportFile.write("Source des images : " + mainDirPath + sourceDirName + imageFileName + "\n")
    exportFile.write("Fichiers conditions initiales : " + mainDirPath + sourceDirName + " > " + SnakeInit_filename + " ; " + "\n")
    exportFile.write("\n")
    exportFile.write("Paramètres du snake\n")
    exportFile.write("nt = " + str(nt) + "\n")
    exportFile.write("stepPlot = " + str(stepPlot) + "\n")
    exportFile.write("blur = " + str(blur) + "\n")
    exportFile.write("A = " + str(A) + "\n")
    exportFile.write("B = " + str(B) + "\n")
    exportFile.write("W_L = " + str(W_L) + "\n")
    exportFile.write("W_E = " + str(W_E) + "\n")
    exportFile.write("Max_It = " + str(Max_It) + "\n")
    exportFile.write("\n")
    exportFile.write("Time = " + str(Tf- Ti))
    exportFile.close()
    
    # Retourner comme résultat les matrices contenant les snakes successifs
    return(SnakeResult)

### 2.4. Post-traitement des snakes : analyse géométrique et enregistrement des résultats

* Fonctions pour fitter un cercle à partir d'une courbe en arc, en utilisant la méthode des moindres carrés. Elles sont utilisées dans les fonctions du bloc suivant.

In [ ]:
def calc_R_circle(x,y, xc, yc):
    """ calculate the distance of each 2D points from the center (xc, yc) """
    return np.sqrt((x-xc)**2 + (y-yc)**2)

def f_circle(c, x, y):
    """ calculate the algebraic distance between the data points and the mean circle centered at c=(xc, yc) """
    Ri = calc_R_circle(x, y, *c)
    return Ri - Ri.mean()

def leastsq_circle(x,y):
    # coordinates of the barycenter
    x_m = np.mean(x)
    y_m = np.mean(y)
    center_estimate = x_m, y_m
    center, ier = optimize.leastsq(f_circle, center_estimate, args=(x,y))
    xc, yc = center
    Ri       = calc_R_circle(x, y, *center)
    R        = Ri.mean()
    residu   = np.sum((Ri - R)**2)
    return xc, yc, R, residu

* Fonctions pour détecter certaines zones de la cellule : le sommet, la base, le morceau de cercle formé par le cytoplasme, et les coins s'en détachant.

In [ ]:
def findHighLow(Snake, a):
    X = Snake[:, 1]
    Y = Snake[:, 0]
    B = Y - a*X
    b_high = np.percentile(B, 2) # the Y axis is upside-down in the images manipulated by python
    b_low = np.percentile(B, 90) # therefore the b_high is in the low percentiles and the opposite for b_low
    return(b_high, b_low)

def fitCircleCytoplasm(Snake, a):
    X, Y = Snake[:, 1], Snake[:, 0]
    B = Y - a*X
    b_high, b_low = findHighLow(Snake, a)
    b_cut = (1*b_high + 2*b_low) / 3
    
    maskSnake = B < b_cut # Once again it's counter-intuitive but the upper part of the snake has a lower y-coordinate
    upperSnake = Snake[maskSnake, :]
    upperX, upperY = upperSnake[:, 1], upperSnake[:, 0]
    
    xc, yc, R, residu = leastsq_circle(upperX, upperY)
    return(xc, yc, R, b_cut)

def fitCircleCorner(subSnake):
    X, Y = subSnake[:, 1], subSnake[:, 0]
    xc, yc, R, residu = leastsq_circle(subSnake)
    return(xc, yc, R, b_cut)

# Fonctions auxiliaires de findCorners() pour détecter les zones du snake proches des angles de contact    

def isAboveBaseLine(y, b, deltaPix = 1):
    test = (b-y >= deltaPix)
    return(test)

def isCloseToBaseLine(y, b_low, b_high):
    test = (y >= 0.6*b_low + 0.4*b_high)
    return(test)

def isPartOfCircle(x, y, xc, yc, R, deltaPix = 1):
    Rxy = ((x-xc)**2 + (y-yc)**2)**0.5
    test = (abs(Rxy-R) <= deltaPix)
    return(test)

def isCloseToCircle(x, y, xc, yc, R, sensitivity = 1.5):
    Rxy = ((x-xc)**2 + (y-yc)**2)**0.5
    test = (Rxy <= sensitivity*R)
    return(test)

def isLeftOrRight(x, xc):
    if(x <= xc):
        side = 'left'
    else:
        side = 'right'
    return(side)

# Fonction qui détecte les points du snake constituant les coins

def findCorners(Snake, a, b_high, b_low, xc, yc, R, b_cut):

    segments = {}
    nPts = Snake.shape[0]
    
    # Etape 1 : trouver des segments de snake qui sont pas trop près du cercle, et pas sur la ligne de base
    # On relève leur indice et leur côté (left/right) et on stocke cela dans un dictionnaire 'segments'.
    prolonge = False
    segmentNum = 0
    listeCourante = []
    side = ''
    for i in range(nPts):
        y, x = Snake[i,0], Snake[i,1]
        
#         print(i)
#         print("Above base line ? " + str(isAboveBaseLine(y, b_low)))
#         print("Close enough to base line ? " + str(isCloseToBaseLine(y, b_low, b_high)))
#         print("Not on the circle ? " + str(not isPartOfCircle(x, y, xc, yc, R)))
#         print("Close enough to circle ? " + str(isCloseToCircle(x, y, xc, yc, R)))
        
        if isAboveBaseLine(y, b_low) and isCloseToBaseLine(y, b_low, b_high) \
        and not isPartOfCircle(x, y, xc, yc, R) and isCloseToCircle(x, y, xc, yc, R):
            if not prolonge:
                segmentNum += 1
                listeCourante = []
                listeCourante.append(i)
                side = isLeftOrRight(x, xc)
                prolonge = True
            elif prolonge:
                listeCourante.append(i)
        else:
            if prolonge:
                prolonge = False
                segments[str(segmentNum)] = {'idx' : listeCourante, 'side' : side}

    # Etape 2 : On colle les suites d'indices qui sont proches les unes des autres en ajoutant les indices manquants
    # (cela correspond au cas où le coin est traversé par un arc du cerle).
    # Puis on allonge symmétriquement les morceaux trouvés pour avoir au moins 10 indices dans chaque.
    # Enfin on supprime ceux dont le ymin est loin de la ligne de base car ce sont probablement des erreurs ; on doit se retrouver avec les tracé des 2 coins !
    
    nSegments = len(segments.keys())
    junctionSize = 3
    i = 1
    while i < nSegments:
        currentSegmentLastElement = segments[str(i)]['idx'][-1]
        nextSegmentFirstElement = segments[str(i+1)]['idx'][0]
        if nextSegmentFirstElement - currentSegmentLastElement <= junctionSize:
            junctionFirstElement = segments[str(i)]['idx'][0]
            junctionLastElement = segments[str(i+1)]['idx'][-1]
            junctionSide = segments[str(i)]['side']
            segments.pop(str(i), None)
            segments.pop(str(i+1), None)
            segments[str(i)] = {'idx' : [i for i in range(junctionFirstElement,junctionLastElement+1)], 'side' : side}  
            i += 1
        i+=1
    
    for k in segments.keys():
        currentSegment = segments[k]
        currentSegmentSize = len(currentSegment['idx'])
        parite = currentSegmentSize % 2
        goalSize = 12 - parite
        appendiceSize = (goalSize - currentSegmentSize)//2
        currentSegment['idx'] = [currentSegment['idx'][0] - i for i in range(appendiceSize+1,1,-1)] \
                                + currentSegment['idx'] \
                                + [currentSegment['idx'][-1] + i for i in range(1,appendiceSize+1)]
#     print(segments)
    minY = {}
    for k in segments.keys():
        currentSegment = segments[k]
        currentListeY = Snake[currentSegment['idx'],0]
        minY[np.min(currentListeY)] = k

    segmentsResult = {'left' : [], 'right' : []}
    
    try:
        FirstBestMinY = np.max(list(minY.keys()))
        FirstGoodSegment = minY.pop(FirstBestMinY)
        segmentsResult[segments[FirstGoodSegment]['side']] = segments[FirstGoodSegment]['idx']
        SecondBestMinY = np.max(list(minY.keys()))
        SecondGoodSegment = minY.pop(SecondBestMinY)
        segmentsResult[segments[SecondGoodSegment]['side']] = segments[SecondGoodSegment]['idx']
    except:
        print('too bad')
        
    return(segmentsResult)

* Fonctions d'analyse : en utilisant la géométrie de la cellule telle qu'on l'a détectée avec les fonctions précédente, ces fonctions vont calculer les paramètres d'intérêts et les sauvegarder dans un fichier texte

In [ ]:
# Fonctions auxiliaires de analyseSnake() 

def computeCytoplasmContact(Snake, xc, yc, R, a, b_low):
    b_low -= 1 # 1 pixel au-dessus
    Ycontact = b_low
    try:
        Rcontact = math.sqrt(R**2 - (b_low-yc)**2)
    except:
        Rcontact = 0
    XcontactLeft = xc - Rcontact
    XcontactRight = xc + Rcontact
    leftCytoContactPoint = np.array([Ycontact, XcontactLeft])
    rightCytoContactPoint = np.array([Ycontact, XcontactRight])
    return(leftCytoContactPoint, rightCytoContactPoint)
    
def computeCellContact(Snake, b_low, xc, yc, R, returnIndices = False):
    nPts = Snake.shape[0]
    # Left side
    i = 0
    y, x = Snake[i,0], Snake[i,1]
    while i < nPts and not (isAboveBaseLine(y, b_low, deltaPix = 0) and isCloseToCircle(x, y, xc, yc, R, sensitivity = 1.4)):
        i += 1
        y, x = Snake[i,0], Snake[i,1]
    leftCellContactPoint = Snake[i,:]
    iLeft = i
    # Right side
    i = nPts-1
    y, x = Snake[i,0], Snake[i,1]
    while i >= 0 and not (isAboveBaseLine(y, b_low, deltaPix = 0) and isCloseToCircle(x, y, xc, yc, R, sensitivity = 1.4)):
        i -= 1
        y, x = Snake[i,0], Snake[i,1]
    rightCellContactPoint = Snake[i,:]
    iRight = i
    # Renvoie les coordonnées ou les indices des points
    if returnIndices:
        return(iLeft, iRight)
    else:
        return(leftCellContactPoint, rightCellContactPoint)
        
def computeCellArea(Snake, b_low, xc, yc, R):
    leftPointIdx, rightPointIdx = computeCellContact(Snake, b_low, xc, yc, R, returnIndices = True)
    Area = 0
    for i in range(leftPointIdx, rightPointIdx-1):
        Area += (0.5*abs(Snake[i,0]-b_low) + 0.5*abs(Snake[i+1,0]-b_low)) * abs(Snake[i+1,1]-Snake[i,1])
        # Trapezoide. Aire = (0.5*H1 + 0.5*H2) * Lbase ; On rappelle que les snakes contiennent les coordonnées en YX.
    return(Area)

def computeHpeel(Snake, corners, b):
    #
    leftCornerY = Snake[corners['left'],0]
    idxLeft = np.argmin(leftCornerY)
    HpeelLeft = abs(b - Snake[corners['left'][idxLeft],0])
    #
    rightCornerY = Snake[corners['right'],0]
    idxRight = np.argmin(rightCornerY)
    HpeelRight = abs(b - Snake[corners['right'][idxRight],0])
        
    return(HpeelLeft, HpeelRight)

Fonction d'analyse principale. Elle appelle toutes les autres pour analyser la série de snake, et sauvegarder les résultats sous forme d'image et d'un fichier texte contenant les résultats numériques.

Definition des différents paramètres géométriques sauvegardés, dans l’ordre du fichier .txt résultat

- T : n° de l’image dans la séquence
- H : hauteur de la cellule au dessus de la lame
- LadhCyto : longueur d’adhésion du cytoplasme, définie à partir du cercle fitté
- LadhCell : longueur d’adhésion totale, définie à partir des deux coins
- AreaSnake : aire comprise entre le snake et la lame
- Rcyto :  rayon de courbure du cercle fittant le cytoplasme
- Rleft & Rright : rayons de courbures des cercles fittant les coins
- HpeelLeft & HpeelRight : hauteur des points de pelage des coins gauche et droit.
- XcCyto, YcCyto, XcLeft, YcLeft, XcRight, YcRight : les positions des centres des 3 cercles définis. Pas là pour être analysés (pas vraiment de sens physique) mais pour permettre la reconstruction géométrique.

In [ ]:
def analyseSnake(scale = 1, dt = 1, showPlot = False):
    imagePath = mainDirPath + sourceDirName + imageFileName + ext
    imageStack = io.imread(imagePath)
    if len(imageStack.shape) == 2:
        nt = 1
        nx = imageStack.shape[1]
        ny = imageStack.shape[0]
    elif len(imageStack.shape) == 3:
        nt = imageStack.shape[0]
        nx = imageStack.shape[2]
        ny = imageStack.shape[1]
    
    resultsSubdirName = 'Results_' + imageFileName + '//'
    quantitativeResultsPath = mainDirPath + resultsDirName + resultsSubdirName + "quantitative_results//"
    snakeFileName = 'Snake.txt'
    allSnakes = extractSnakeFromFile(quantitativeResultsPath, snakeFileName)
    
    create_dir(resultsSubdirName + "analyseImages")
    
    analyseResultFile = open(quantitativeResultsPath + 'Analyse' + '.txt', "w")
    Header = "Film analysé  :\t" + imagePath + "\tNombre d'images :\t" + str(nt) + "\n" \
    + "Resultats de l'analyse de la série de snakes. Toutes les longueurs sont en pixel et les durée en frames. " \
    + "Pour faire la conversion, utilisez les valeurs à la ligne suivante.\n" \
    + "Echelle spatiale (px/µm) :\t" + str(scale) + "\tEchelle temporelle (s/frames) :\t" + str(dt) +"\n" \
    + "\n" \
    + "T\tH\tLadhCyto\tLadhCell\tAreaSnake\tRcyto\tRleft\tRright\tHpeelLeft\tHpeelRight\tXcCyto\tYcCyto\tXcLeft\tYcLeft\tXcRight\tYcRight"
    analyseResultFile.write(Header)
    
    a, b, lineAngle, lineDist = houghTransform(imageStack[0])
    
    for T in range(nt):
        image = imageStack[T]
        Snake = allSnakes[T]
        
        # Boundaries
        b_high, b_low = findHighLow(Snake, a)
        # Circle
        XcCyto, YcCyto, Rcyto, b_cut = fitCircleCytoplasm(Snake, a)
        # Corners
        corners = findCorners(Snake, a, b_high, b_low, XcCyto, YcCyto, Rcyto, b_cut)
        
        H = abs(b_high-b_low)
        
        leftCytoContactPoint, rightCytoContactPoint = computeCytoplasmContact(Snake, XcCyto, YcCyto, Rcyto, a, b_low)
        LadhCyto = abs(leftCytoContactPoint[1] - rightCytoContactPoint[1])
        
        leftCellContactPoint, rightCellContactPoint = computeCellContact(Snake, b_low, XcCyto, YcCyto, Rcyto, returnIndices = False)
        LadhCell = abs(leftCellContactPoint[1] - rightCellContactPoint[1])
        
        Area = computeCellArea(Snake, b_low, XcCyto, YcCyto, Rcyto)
        
        XLeftCorner, YLeftCorner = Snake[corners['left'], 1], Snake[corners['left'], 0]
        XcLeft, YcLeft, RLeft, residuLeft = leastsq_circle(XLeftCorner, YLeftCorner)
        
        XRightCorner, YRightCorner = Snake[corners['right'], 1], Snake[corners['right'], 0]
        XcRight, YcRight, RRight, residuRight = leastsq_circle(XRightCorner, YRightCorner)
        
        HpeelLeft, HpeelRight = computeHpeel(Snake, corners, b)
        
        dataLine = "\n" + str(T) + "\t" + str(H) + "\t" + str(LadhCyto) + "\t" + str(LadhCell) + "\t" + str(Area) \
        + "\t" + str(Rcyto) + "\t" + str(RLeft) + "\t" + str(RRight) + "\t" + str(HpeelLeft) + "\t" + str(HpeelRight) \
        + "\t" + str(XcCyto) + "\t" + str(YcCyto) + "\t" + str(XcLeft) + "\t" + str(YcLeft) + "\t" + str(XcRight) + "\t" + str(YcRight)
        analyseResultFile.write(dataLine)
        
        # Trace et sauvegarde l'image
        
        fig, ax = plt.subplots(1, 1, figsize = (20, 10))
        ax.imshow(image)
        ax.plot(Snake[:,1], Snake[:,0], 'b-')
        ax.plot(Snake[corners['left'],1], Snake[corners['left'],0], 'm-', lw = 3)
        ax.plot(Snake[corners['right'],1], Snake[corners['right'],0], 'm-', lw = 3)
        ax.plot([0, nx-1], [b, a*(nx-1)+b], 'r')
        ax.plot([0, nx-1], [b_low, a*(nx-1)+b_low], 'm--')
        ax.plot([leftCytoContactPoint[1]], [leftCytoContactPoint[0]], 'go', markersize=12)
        ax.plot([rightCytoContactPoint[1]], [rightCytoContactPoint[0]], 'go', markersize=12)
        ax.plot([leftCellContactPoint[1]], [leftCellContactPoint[0]], 'go', markersize=12)
        ax.plot([rightCellContactPoint[1]], [rightCellContactPoint[0]], 'go', markersize=12)
        theta_fit = np.linspace(-np.pi, np.pi, 180)
        x_fit = XcCyto + Rcyto*np.cos(theta_fit)
        y_fit = YcCyto + Rcyto*np.sin(theta_fit)
        ax.plot(x_fit, y_fit, 'y--')
        ax.plot([XcCyto], [YcCyto], 'yo')
        x_fit = XcLeft + RLeft*np.cos(theta_fit)
        y_fit = YcLeft + RLeft*np.sin(theta_fit)
        ax.plot(x_fit, y_fit, 'w--')
        ax.plot([XcLeft], [YcLeft], 'wo')
        x_fit = XcRight + RRight*np.cos(theta_fit)
        y_fit = YcRight + RRight*np.sin(theta_fit)
        ax.plot(x_fit, y_fit, 'w--')
        ax.plot([XcRight], [YcRight], 'wo')
        ax.set_xlim([0,nx])
        ax.set_ylim([ny,0])
        saveimage_path = resultsSubdirName + "analyseImages//" + "cellGeometry" + "_" + str(T) + ".png"
        fig.savefig(saveimage_path)
        
        if not showPlot:
            plt.close('all')
    
    analyseResultFile.close()

## 3. Scripts à exécuter

C'est ici qu'on appelera toutes les fonctions définies plus haut.

### 3.1. Paramètres

Ici on précise 'scale' l'échelle des images en px/µm.


L'arborescence de dossiers y est indiquée aussi.
J'ai fonctionné avec un dossier principal indiqué par "mainDirPath",
qui contient les sous-dossiers :
- "RawImages"  - images sans traitements. Pas appelé par le code python.
- "CropedImages" - images croppées par IJ et snakes initiaux tracés en IJ correspondants.
- "CorrectedImages" - images traitées par le prétraitement et snakes initiaux corrigés correspondants.
- "SnakeResults" - résultats. Ce dossier est lui-même organisé en sous-dossiers, un par stack d'image analysé, qui contiennent :
    - ActiveContourData.txt : résumé des paramètres de tracé du snake.
    - "exampleImages" qui contient les images exemples pour le tracé du snake.
    - "analyseImages" qui contient les images où sont tracés les figures géométriques fittées par l'analyse.
    - "quantitative_results" qui contient les fichier Snake.txt (les coordonnées des snakes) et Analyse.txt (les paramètres fittés par la fonction d'analyse).

In [ ]:
# scale parameter: 1µm = 'scale' pixels
scale = 4.6154 
# scale = 13.7362 # Pour un autre objectif

# Directories organisation
mainDirPath = 'C://Users//josep//Desktop//MSC_Films_Jon//'
rawImagesDirName = 'CropedImages//'
treatedImagesDirName = 'CorrectedImages//'
resultsDirName = 'SnakeResults//'

### 3.2. Prétraitement

In [ ]:
rawImageName = 'Spreading_20-06-04-m1-b_cropped'
treatedImageName = 'Spreading_20-06-04-m1-b_denoised'
ext = '.tif'

# Emplacement du fichier
rawImagePath = mainDirPath + rawImagesDirName + rawImageName + ext
imageStack = io.imread(rawImagePath)

# Emplacement du snake initial
SnakeInit_dirpath = mainDirPath + rawImagesDirName
SnakeInit_filename = rawImageName + '_SnakeInit' + '.txt'
SnakeInit_YX = extractSnakeFromFile(SnakeInit_dirpath, SnakeInit_filename, header = False, scale = scale, reverseXY = True)[0]

# Remarque : pour faire le snake initial, faire un tracé approximatif sur imageJ 
# avec l'outil "Freehand Line" et le sauvegarder avec "File > Save as > Position XY" 
# sous forme d'un .txt de points.

SnakeInit_YX # Vérifier qu'on a un array de nombres entiers, à un arrondi près : il s'agit de positions XY !

/!\ L'étape de prétraitement ci-dessous est de loin la plus chronophage du code, je conseille de la commenter lorsqu'on a pas prévu de s'en servir pour ne pas la lancer par erreur.

In [ ]:
denoisedImageStack, rotatedSnakeInit_YX = pretraitement(imageStack, SnakeInit_YX, referenceSide = 'left&right', 
                    saveDirPath = mainDirPath + treatedImagesDirName, saveFileName = treatedImageName)

fig, ax = plt.subplots(1,1,figsize=(10,5))
ax.imshow(denoisedImageStack[0])
ax.plot(rotatedSnakeInit_YX[:,1], rotatedSnakeInit_YX[:,0], 'r--')

### 3.3. Tracé des snakes

In [ ]:
# Emplacement du fichier
sourceDirName = 'CorrectedImages//'
imageFileName = 'Spreading_20-06-04-m1-b_denoised'
ext = '.tif'

# Emplacement du snake initial
SnakeInit_filename = imageFileName + '_rotatedSnakeInit' + '.txt'

S = makeSnake(A = 0.02, B = 40, W_L = -20, W_E = 200, B_C = 'fixed', Max_It = 500, blur = 1, stepPlot = 5, SnakePoints = 100, showPlot = True)
# Best parms for film_ : makeSnake(A = 0.2, B = 150, W_L = -120, W_E = 320, B_C = 'fixed', Max_It = 1000, blur = 1, stepPlot = 5, SnakePoints = 100)
# Best parms for film_a - 30 first frames only : makeSnake(A = 0.05, B = 100, W_L = -75, W_E = 125, B_C = 'fixed', Max_It = 1000, blur = 1, stepPlot = 5, SnakePoints = 100, showPlot = True)
# Best parms for film_a - 30 first frames only V2 : makeSnake(A = 0.1, B = 150, W_L = -80, W_E = 150, B_C = 'fixed', Max_It = 500, blur = 2, stepPlot = 5, SnakePoints = 100, showPlot = True)
# Best parms for film_b : makeSnake(A = 0.1, B = 100, W_L = 0, W_E = 300, B_C = 'fixed', Max_It = 1000, blur = 1, stepPlot = 5, SnakePoints = 100)
# Best parms for film_d : makeSnake(A = 0.1, B = 15, W_L = -25, W_E = 350, B_C = 'fixed', Max_It = 1000, blur = 2, stepPlot = 5, SnakePoints = 100)

### 3.4. Analyse des snakes

In [ ]:
# dirname = 'C://Users//josep//Desktop//MSC_Films_Jon//CropedImages//'

# imagesPath = "C://Users//josep//Desktop//MSC_Films_Jon//CropedImages//" \
#             + "Spreading_20-02-14-m3-a_BF_denoised.tif"
# resultsDirPath = "C://Users//josep//Desktop//MSC_Films_Jon//SnakeResults//" \
#                 + "Result of Spreading_20-02-14-m3-a_BF_denoised//"

sourceDirName = 'CorrectedImages//'
imageFileName = 'Spreading_20-06-04-m1-b_denoised'
ext = '.tif'

analyseSnake(scale = scale, dt = 1, showPlot = True)

## ET VOILA !

## ***
## APPENDICE : Fonctions de test

In [ ]:
def test_houghTransform(dirname, filename, ext):
    imgpath = os.path.join(dirname, filename + ext)
    image = io.imread(imgpath)
    image_1 = image[0]
    a, b, lineAngle, lineDist = houghTransform(image_1, plot = True)
    print(lineAngle)


dirname = 'C://Users//josep//Desktop//MSC_Films_Jon//CropedImages//'
ext = '.tif'


filename = 'Spreading_20-02-14-m3-a_BF_crop'
test_houghTransform(dirname, filename, ext)

filename = 'Spreading_20-02-14-m3-b_BF_crop'
test_houghTransform(dirname, filename, ext)

filename = 'Spreading_20-02-14-m3-d_BF_crop'
test_houghTransform(dirname, filename, ext)

filename = 'Spreading_20-06-04-m1-b_cropped'
test_houghTransform(dirname, filename, ext)

In [ ]:
def test_makeHorizontal(dirname, filename, SnakeInit_filename, ext):
    # Get image
    imgpath = os.path.join(dirname, filename + ext)
    image = io.imread(imgpath)
    imageStack = np.array([image[0]])
    # Get initial snake
    SnakeInit_path = os.path.join(dirname, SnakeInit_filename)
    SnakeInit_file = open(SnakeInit_path, 'r')
    SnakeInit_lines = SnakeInit_file.readlines()
    n_Snake = len(SnakeInit_lines)
    SnakeInit_X = np.zeros(n_Snake)
    SnakeInit_Y = np.zeros(n_Snake)
    for i in range(0, n_Snake):
        line = SnakeInit_lines[i].split('\t')
        SnakeInit_X[i] = int(float(line[0]) * scale)
        SnakeInit_Y[i] = int(float(line[1]) * scale)
    SnakeInit_Contour = np.array([SnakeInit_Y, SnakeInit_X]).T
    
    a, b, lineAngle, lineDist = houghTransform(imageStack[0])
    rotatedImageStack, rotatedSnakeInit_Contour = makeHorizontal(imageStack, lineAngle, SnakeInit_Contour) # 
    n_images = rotatedImageStack.shape[0]
    fig, ax = plt.subplots()
    for i in range(n_images):
        ax.imshow(rotatedImageStack[i])
        if rotatedSnakeInit_Contour.shape[0] > 0:
            ax.plot(rotatedSnakeInit_Contour[:, 1], rotatedSnakeInit_Contour[:, 0], '--b', lw=2)

        
dirname = 'C://Users//josep//Desktop//MSC_Films_Jon//CropedImages//'
ext = '.tif'

filename = 'Spreading_20-02-14-m3-a_BF_crop'
SnakeInit_filename = filename + '_SnakeInit' + '.txt'
test_makeHorizontal(dirname, filename, SnakeInit_filename, ext)

filename = 'Spreading_20-02-14-m3-b_BF_crop'
SnakeInit_filename = filename + '_SnakeInit' + '.txt'
test_makeHorizontal(dirname, filename, SnakeInit_filename, ext)

filename = 'Spreading_20-02-14-m3-d_BF_crop'
SnakeInit_filename = filename + '_SnakeInit2' + '.txt'
test_makeHorizontal(dirname, filename, SnakeInit_filename, ext)

In [ ]:
def test_denoisePlateShadow(dirname, filename, ext):
    imgpath = os.path.join(dirname, filename + ext)
    imageStack = io.imread(imgpath)
    oneImageStack = np.array([imageStack[10]])
    a, b, lineAngle, lineDist = houghTransform(oneImageStack[0])
    imageHorizontal = makeHorizontal(oneImageStack, lineAngle)
    imageInput = imageHorizontal[0,20:-20,25:-25]
    denoisedImage = denoisePlateShadow(imageInput)
    fig, ax = plt.subplots(3, 1, figsize = (10, 10))
    ax[0].imshow(oneImageStack[0])
    ax[1].imshow(imageInput)
    ax[2].imshow(denoisedImage)
    plt.tight_layout()
        
dirname = 'C://Users//josep//Desktop//MSC_Films_Jon//CropedImages//'
ext = '.tif'

filename = 'Spreading_20-02-14-m3-a_BF_crop'
test_denoisePlateShadow(dirname, filename, ext)

In [ ]:
def test_pretraitement(dirname, filename, SnakeInit_filename, ext):
    # Get image
    imgpath = os.path.join(dirname, filename + ext)
    imageStack = io.imread(imgpath)
    imageStack = imageStack[:10]
    # Get initial snake
    SnakeInit_path = os.path.join(dirname, SnakeInit_filename)
    SnakeInit_file = open(SnakeInit_path, 'r')
    SnakeInit_lines = SnakeInit_file.readlines()
    n_Snake = len(SnakeInit_lines)
    SnakeInit_X = np.zeros(n_Snake)
    SnakeInit_Y = np.zeros(n_Snake)
    for i in range(0, n_Snake):
        line = SnakeInit_lines[i].split('\t')
        SnakeInit_X[i] = int(float(line[0]) * scale)
        SnakeInit_Y[i] = int(float(line[1]) * scale)
    SnakeInit_YX = np.array([SnakeInit_Y, SnakeInit_X]).T
    ###
    treatedImageStack, treatedSnakeInit_YX = pretraitement(imageStack, SnakeInit_YX, 12, 'right')
    ###
    fig, ax = plt.subplots(1, 3, figsize = (30, 5))
    ax[0].imshow(imageStack[0])
    ax[0].plot(SnakeInit_YX[:, 1], SnakeInit_YX[:, 0], 'r--')
    ax[1].imshow(treatedImageStack[0])
    ax[1].plot(treatedSnakeInit_YX[:, 1], treatedSnakeInit_YX[:, 0], 'r--')
    ax[2].imshow(makeFilter(treatedImageStack[0], 2))
    ax[2].plot(treatedSnakeInit_YX[:, 1], treatedSnakeInit_YX[:, 0], 'r--')
    
    for i in range(1,10):
        fig, ax = plt.subplots(1, 3, figsize = (30, 5))
        ax[0].imshow(imageStack[i])
        ax[1].imshow(treatedImageStack[i])
        ax[2].imshow(makeFilter(treatedImageStack[i], 2))


dirname = 'C://Users//josep//Desktop//MSC_Films_Jon//CropedImages//'
ext = '.tif'
filename = 'Spreading_20-02-14-m3-a_BF_crop'
SnakeInit_filename = filename + '_SnakeInit2' + '.txt'

test_pretraitement(dirname, filename, SnakeInit_filename, ext)


In [ ]:
def test_findHighLow(dirname, filename, ext, allSnakes, index):
    imgpath = os.path.join(dirname, filename + ext)
    image = io.imread(imgpath)
    image_i = image[index]
    Snake_i = allSnakes[index]
    a, b, lineAngle, lineDist = houghTransform(image_i)
    b_high, b_low = findHighLow(Snake_i, a)
    ny = image_i.shape[0]
    nx = image_i.shape[1]
    fig, ax = plt.subplots(1,1, figsize = (10, 10))
    ax.imshow(image_i)
    ax.plot(Snake_i[:, 1], Snake_i[:, 0], 'b-')
    ax.plot([0, nx-1], [b, a*(nx-1)+b], 'r-')
    ax.plot([0, nx-1], [b_low, a*(nx-1)+b_low], 'g--')
    ax.plot([0, nx-1], [b_high, a*(nx-1)+b_high], 'g--')

In [ ]:
def test_fitCircleCytoplasm(dirname, filename, ext, allSnakes, index):
    imgpath = os.path.join(dirname, filename + ext)
    imageStack = io.imread(imgpath)
    denoisedImageStack = pretraitement(imageStack, margin = 10, referenceSide = 'left&right')
    image_i = denoisedImageStack[index]
    Snake_i = allSnakes[index]
    # Plate position
    a, b, lineAngle, lineDist = houghTransform(image_i)
    # Boundaries
    b_high, b_low = findHighLow(Snake_i, a)
    # Circle
    xc, yc, R, b_cut = fitCircleCytoplasm(Snake_i, a)
    theta_fit = np.linspace(-np.pi, np.pi, 180)
    x_fit = xc + R*np.cos(theta_fit)
    y_fit = yc + R*np.sin(theta_fit)
    
    # Plot_results
    ny = image_i.shape[0]
    nx = image_i.shape[1]
    fig, ax = plt.subplots(1,1, figsize = (10, 10))
    ax.imshow(image_i)
    ax.plot(Snake_i[:, 1], Snake_i[:, 0], 'b-')
    ax.plot([0, nx-1], [b, a*(nx-1)+b], 'r-')
    ax.plot([0, nx-1], [b_low, a*(nx-1)+b_low], 'g--')
    ax.plot([0, nx-1], [b_high, a*(nx-1)+b_high], 'g--')
    ax.plot([0, nx-1], [b_cut, a*(nx-1)+b_cut], 'w--')
    ax.plot(x_fit, y_fit, 'y--' , lw=2)
    ax.plot([xc], [yc], 'yo')
    
#     circle_pic_path = os.path.join(savedirname, saveresultname, "circle_pic")
#     create_dir(circle_pic_path)
#     savecircle_path = os.path.join(savedirname, saveresultname, "circle_pic", savepicname + str(index) + savepicExt)
#     fig.savefig(savecircle_path)



filename = 'Spreading_20-02-14-m3-d_BF_crop'
SnakeInit_filename = filename + '_SnakeInit2' + '.txt'
allSnakes = makeSnake(A = 0.1, B = 100, W_L = -65, W_E = 125, B_C = 'fixed', Max_It = 1000, blur = 1, stepPlot = 1, SnakePoints = 100, showPlot = False)
for index in range(0,30,1):
    test_fitCircle(dirname, filename, ext, allSnakes, index)

# filename = 'Spreading_20-02-14-m3-b_BF_crop'
# SnakeInit_filename = filename + '_SnakeInit' + '.txt'
# allSnakes = makeSnake(A = 0.1, B = 100, W_L = 0, W_E = 300, B_C = 'fixed', Max_It = 1000, blur = 1, stepPlot = 5, SnakePoints = 100, showPlot = False)
# test_findHighLow(dirname, filename, ext, allSnakes, index)

# filename = 'Spreading_20-02-14-m3-d_BF_crop'
# SnakeInit_filename = filename + '_SnakeInit2' + '.txt'
# allSnakes = makeSnake(A = 0.1, B = 15, W_L = -50, W_E = 325, B_C = 'fixed', Max_It = 1000, blur = 2, stepPlot = 5, SnakePoints = 100, showPlot = False)
# for index in range(0,31,1):
#     test_fitCircle(dirname, filename, ext, allSnakes, index)

# Best parms for film_a : makeSnake(A = 0.2, B = 150, W_L = -120, W_E = 320, B_C = 'fixed', Max_It = 1000, blur = 1, stepPlot = 5, SnakePoints = 100)
# Best parms for film_b : makeSnake(A = 0.1, B = 100, W_L = 0, W_E = 300, B_C = 'fixed', Max_It = 1000, blur = 1, stepPlot = 5, SnakePoints = 100)
# Best parms for film_d : makeSnake(A = 0.1, B = 15, W_L = -25, W_E = 350, B_C = 'fixed', Max_It = 1000, blur = 2, stepPlot = 5, SnakePoints = 100)

In [ ]:
def test_findCorners(imageStack, allSnakes, N = 30):
    # Plate position
    a, b, lineAngle, lineDist = houghTransform(imageStack[0])
    # a = 0
    
    for i in range(N):
        image = imageStack[i]
        Snake = allSnakes[i]
        # Boundaries
        b_high, b_low = findHighLow(Snake, a)
        # Circle
        xc, yc, R, b_cut = fitCircleCytoplasm(Snake, a)
        
        segments = findCorners(Snake, a, b_high, b_low, xc, yc, R, b_cut)
        print(segments)
        
        theta_fit = np.linspace(-np.pi, np.pi, 180)
        x_fit = xc + R*np.cos(theta_fit)
        y_fit = yc + R*np.sin(theta_fit)

        # Plot_results
        ny = image.shape[0]
        nx = image.shape[1]
        fig, ax = plt.subplots(1,1, figsize = (10, 5))
        ax.imshow(image)
        ax.plot(Snake[:, 1], Snake[:, 0], 'b-')
        ax.plot([0, nx-1], [b, a*(nx-1)+b], 'r-')
        ax.plot([0, nx-1], [b_low, a*(nx-1)+b_low], 'g--')
        ax.plot([0, nx-1], [b_high, a*(nx-1)+b_high], 'g--')
        ax.plot([0, nx-1], [b_cut, a*(nx-1)+b_cut], 'g--')
        ax.plot(x_fit, y_fit, 'y--' , lw=2)
        ax.plot([xc], [yc], 'yo')

        ax.plot(Snake[segments['right'], 1], Snake[segments['right'], 0], 'mo')
        ax.plot(Snake[segments['left'], 1], Snake[segments['left'], 0], 'mo')
        
dirname = 'C://Users//josep//Desktop//MSC_Films_Jon//CorrectedImages//'
ext = '.tif'
filename = 'Spreading_20-06-04-m1-b_denoised'
snakeDir = "C://Users//josep//Desktop//MSC_Films_Jon//SnakeResults//Results_" + filename + "//quantitative_results//"
snakeFileName = 'Snake' + '.txt'
allSnakes = extractSnakeFromFile(snakeDir, snakeFileName)
imgpath = os.path.join(dirname, filename + ext)
imageStack = io.imread(imgpath)

test_findCorners(imageStack, allSnakes, N = 5)